# Histogram of numbers between 0 and 1

A histogram counts how many of a set of values fall into each of `nbins` bins of equal width
that together cover the interval from 0 to 1. A value `v` belongs to bin `floor(v * nbins)`.
Many values can belong to the same bin, so the counts are updated in unpredictable places, and
two workers that update the same count at the same time can lose an increment.

In [ ]:
import numpy as np


def histogram(x, nbins):
    """Count the values of `x`, which lie in [0, 1), in each of `nbins` equal bins.

    A value v belongs to bin floor(v * nbins). If the product v * nbins rounds up to nbins, the
    value belongs to the last bin.
    """
    bins = np.minimum((x * nbins).astype(np.int64), nbins - 1)
    return np.bincount(bins, minlength=nbins)

In [ ]:
print(histogram(np.array([0.05, 0.3, 0.35, 0.8, 0.99]), 4))

The declaration below tells the harness how to call the function and how to check a routine
against it. The number of bins `nbins` is both an argument and the length of the result, and the
C routine takes the values, their number `n`, the array of counts and then `nbins`, in that
order. The counts are whole numbers, so a routine must reproduce them exactly. `make_inputs`
chooses the number of bins by the square-root rule, the square root of the number of values
rounded up, but at most 1024. It draws values between 0 and 1 whose density rises or falls
towards one end. A few of the values are set to the lowest value, to the highest value below 1,
or to the exact edge of a bin.

In [ ]:
def make_inputs(size, rng):
    nbins = int(min(1024, np.ceil(np.sqrt(size))))
    x = rng.random(size) ** rng.uniform(0.5, 2.0)
    special = rng.random(size) < 0.05
    edges = np.concatenate(([0.0, np.nextafter(1.0, 0.0)], np.arange(nbins) / nbins))
    x[special] = rng.choice(edges, special.sum())
    return {"x": x, "nbins": nbins}


ROUTINE = {
    "function": histogram,
    "make_inputs": make_inputs,
    "arguments": {"x": "float64[n]", "nbins": "int32"},
    "results": {"counts": "int64[nbins]"},
    "c_order": ["x", "n", "counts", "nbins"],
    "test_sizes": [1, 1_000, 100_000, 3_000_000],
    "benchmark_size": 100_000_000,
    "tolerance": 0,
}